In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import shutil, os
from pathlib import Path

DATASET = Path('/kaggle/input/datasets/tamikshasharma1320/ecg-codes-kaggle')
WORK = Path('/kaggle/working/ecg_project')
WORK.mkdir(exist_ok=True, parents=True)

# Copy code files and CSVs
for f in ['config_OPTIMIZED.py', 'model_OPTIMIZED.py', 'train_OPTIMIZED.py',
          'pre_render_OPTIMIZED.py', 'ptbxl_database.csv', 'scp_statements.csv']:
    shutil.copy(DATASET / f, WORK / f)
    print(f'  Copied {f}')

# Copy images from nested folder
src_imgs = DATASET / 'ecg_images' / 'ecg_images'
dst_imgs = WORK / 'ecg_images'
if dst_imgs.exists():
    shutil.rmtree(dst_imgs)
shutil.copytree(str(src_imgs), str(dst_imgs))
print(f'  Copied {len(list(dst_imgs.glob("*.png")))} images')

# Create required directories
for d in ['models', 'logs', 'checkpoints', 'uploads']:
    (WORK / d).mkdir(exist_ok=True)

print('\n[OK] ecg_project folder ready!')

In [ ]:
config_lines = [
    "import os, sys, torch, logging",
    "from pathlib import Path",
    "",
    "if sys.platform == 'win32':",
    "    try:",
    "        sys.stdout.reconfigure(encoding='utf-8', errors='replace')",
    "        sys.stderr.reconfigure(encoding='utf-8', errors='replace')",
    "    except Exception:",
    "        pass",
    "",
    "DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')",
    "IS_CPU  = DEVICE.type == 'cpu'",
    "IS_CUDA = DEVICE.type == 'cuda'",
    "print(f'Device: {DEVICE} | CUDA: {torch.cuda.is_available()}')",
    "",
    "BASE_DIR        = Path('/kaggle/working/ecg_project')",
    "ECG_IMAGE_DIR   = BASE_DIR / 'ecg_images'",
    "RECORDS_DIR     = BASE_DIR / 'records500'",
    "MODELS_DIR      = BASE_DIR / 'models'",
    "LOGS_DIR        = BASE_DIR / 'logs'",
    "CHECKPOINTS_DIR = BASE_DIR / 'checkpoints'",
    "UPLOAD_FOLDER   = BASE_DIR / 'uploads'",
    "DB_CSV          = BASE_DIR / 'ptbxl_database.csv'",
    "SCP_CSV         = BASE_DIR / 'scp_statements.csv'",
    "LOG_FILE        = LOGS_DIR / 'training.log'",
    "",
    "for _d in [MODELS_DIR, LOGS_DIR, CHECKPOINTS_DIR, UPLOAD_FOLDER]:",
    "    _d.mkdir(exist_ok=True, parents=True)",
    "",
    "logging.basicConfig(",
    "    level=logging.INFO,",
    "    format='%(asctime)s - %(levelname)s - %(message)s',",
    "    handlers=[",
    "        logging.FileHandler(LOG_FILE, encoding='utf-8'),",
    "        logging.StreamHandler(sys.stdout),",
    "    ],",
    ")",
    "logger = logging.getLogger(__name__)",
    "for _h in logging.getLogger().handlers:",
    "    if isinstance(_h, logging.StreamHandler):",
    "        try:",
    "            _h.stream.reconfigure(errors='replace')",
    "        except Exception:",
    "            pass",
    "",
    "SAMPLING_RATE       = 100",
    "INPUT_SIGNAL_LEN    = 1000",
    "OUTPUT_WAVEFORM_LEN = 500",
    "NUM_LEADS           = 12",
    "",
    "LATENT_DIM             = 512",
    "NUM_CLASSES            = 5",
    "NHEAD                  = 8",
    "NUM_TRANSFORMER_LAYERS = 3",
    "IMAGE_SIZE             = 224",
    "NORM_MEAN = [0.485, 0.456, 0.406]",
    "NORM_STD  = [0.229, 0.224, 0.225]",
    "",
    "if IS_CUDA:",
    "    BATCH_SIZE     = 16",
    "    VAL_BATCH_SIZE = 32",
    "    NUM_WORKERS    = 4",
    "    PIN_MEMORY     = True",
    "    NUM_EPOCHS     = 10",
    "    WARMUP_EPOCHS  = 3",
    "else:",
    "    BATCH_SIZE     = 2",
    "    VAL_BATCH_SIZE = 4",
    "    NUM_WORKERS    = 0",
    "    PIN_MEMORY     = False",
    "    NUM_EPOCHS     = 6",
    "    WARMUP_EPOCHS  = 2",
    "",
    "FINETUNE_EPOCHS = NUM_EPOCHS - WARMUP_EPOCHS",
    "TOTAL_EPOCHS    = NUM_EPOCHS",
    "",
    "LEARNING_RATE      = 1e-4",
    "FINETUNE_LR        = 1e-6",
    "LOSS_ALPHA         = 0.7",
    "LOSS_BETA          = 0.3",
    "WEIGHT_DECAY       = 1e-4",
    "GRADIENT_CLIP      = 1.0",
    "ACCUMULATION_STEPS = 2 if IS_CPU else 1",
    "MAX_RECORDS        = 500 if IS_CPU else None",
    "",
    "VALIDATION_SPLIT        = 0.15",
    "TEST_SPLIT              = 0.15",
    "EARLY_STOPPING_PATIENCE = 4",
    "SAVE_BEST_ONLY          = True",
    "",
    "MEMORY_CLEANUP_FREQ        = 5",
    "USE_AMP                    = IS_CUDA",
    "USE_GRADIENT_CHECKPOINTING = IS_CPU",
    "",
    "LOG_INTERVAL      = 5",
    "VERBOSE           = True",
    "USE_PROGRESS_BARS = True",
    "",
    "CLASS_NAMES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']",
    "CLASS_DESCRIPTIONS = {",
    "    'NORM': 'Normal ECG',",
    "    'MI'  : 'Myocardial Infarction',",
    "    'STTC': 'ST/T-Wave Change',",
    "    'CD'  : 'Conduction Disturbance',",
    "    'HYP' : 'Hypertrophy',",
    "}",
    "CLASS_MAP          = {name: idx for idx, name in enumerate(CLASS_NAMES)}",
    "SCP_SUPERCLASS_COL = 'diagnostic_class'",
    "",
    "def verify_paths():",
    "    logger.info('='*70)",
    "    logger.info('VERIFYING PATHS')",
    "    logger.info('='*70)",
    "    checks = {",
    "        'ECG Images'        : ECG_IMAGE_DIR,",
    "        'ptbxl_database.csv': DB_CSV,",
    "        'scp_statements.csv': SCP_CSV,",
    "    }",
    "    all_ok = True",
    "    for name, path in checks.items():",
    "        exists = path.exists()",
    "        logger.info(f\"[{'OK' if exists else 'MISSING'}] {name}: {path}\")",
    "        if not exists:",
    "            all_ok = False",
    "    logger.info('='*70)",
    "    return all_ok",
    "",
    "def print_config():",
    "    logger.info('='*70)",
    "    logger.info('CONFIGURATION SUMMARY')",
    "    logger.info('='*70)",
    "    logger.info(f'Device: {DEVICE} | CPU Mode: {IS_CPU}')",
    "    logger.info(f'Batch Size: {BATCH_SIZE} | Workers: {NUM_WORKERS}')",
    "    logger.info(f'Epochs: {NUM_EPOCHS} (Warmup: {WARMUP_EPOCHS}, Finetune: {FINETUNE_EPOCHS})')",
    "    logger.info('Split: 70% train / 15% val / 15% test')",
    "    logger.info(f'Early Stopping Patience: {EARLY_STOPPING_PATIENCE}')",
    "    logger.info(f'Max Records: {MAX_RECORDS}')",
    "    logger.info('='*70)",
]

with open('/kaggle/working/ecg_project/config_OPTIMIZED.py', 'w') as f:
    f.write('\n'.join(config_lines))
print("[OK] config_OPTIMIZED.py written")

In [ ]:
train_code = '''
import os, gc, ast, time, json, sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import logging

from config_OPTIMIZED import (
    DEVICE, IS_CPU, IS_CUDA,
    BATCH_SIZE, VAL_BATCH_SIZE, NUM_WORKERS, PIN_MEMORY,
    NUM_EPOCHS, WARMUP_EPOCHS, TOTAL_EPOCHS,
    LEARNING_RATE, FINETUNE_LR, WEIGHT_DECAY, GRADIENT_CLIP,
    LOSS_ALPHA, LOSS_BETA, ACCUMULATION_STEPS,
    MAX_RECORDS, EARLY_STOPPING_PATIENCE, SAVE_BEST_ONLY,
    MEMORY_CLEANUP_FREQ, USE_PROGRESS_BARS,
    IMAGE_SIZE, NORM_MEAN, NORM_STD,
    OUTPUT_WAVEFORM_LEN, INPUT_SIGNAL_LEN,
    ECG_IMAGE_DIR, RECORDS_DIR, DB_CSV, SCP_CSV, MODELS_DIR,
    BASE_DIR, CLASS_NAMES, SCP_SUPERCLASS_COL,
    logger, print_config, verify_paths,
    VALIDATION_SPLIT, TEST_SPLIT
)
from model_OPTIMIZED import UltimateECGHybrid, build_model

SUPERCLASS_MAP = {"NORM": 0, "MI": 1, "STTC": 2, "CD": 3, "HYP": 4}

# ─────────────────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────────────────

class ECGDataset(Dataset):
    def __init__(self, records, meta, scp_df, transform=None):
        self.records   = records
        self.meta      = meta
        self.scp_df    = scp_df
        self.transform = transform or transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ])
        # Pre-cache labels
        self.labels = [self._get_label(rid) for rid in self.records]

    def _get_label(self, ecg_id):
        try:
            codes = self.meta.loc[ecg_id].scp_codes
            for code, confidence in codes.items():
                if code in self.scp_df.index:
                    sc = self.scp_df.loc[code][SCP_SUPERCLASS_COL]
                    if isinstance(sc, str) and sc in SUPERCLASS_MAP:
                        return SUPERCLASS_MAP[sc]
        except Exception:
            pass
        return 0  # default NORM

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rid = self.records[idx]
        # Load image
        img_path = ECG_IMAGE_DIR / f"{rid}.png"
        try:
            with Image.open(img_path) as img:
                img_t = self.transform(img.convert("RGB"))
        except Exception:
            img_t = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)

        # Load waveform target
        try:
            raw = str(self.meta.loc[rid].filename_hr).replace("\\\\", "/")
            if "records500/" in raw:
                fname = raw.split("records500/", 1)[-1]
            else:
                fname = raw.split("/")[-1]
            import wfdb
            rpath  = RECORDS_DIR / fname
            signal = wfdb.rdrecord(str(rpath)).p_signal[:, 0]
            total  = INPUT_SIGNAL_LEN + OUTPUT_WAVEFORM_LEN
            if len(signal) < total:
                signal = np.pad(signal, (0, total - len(signal)))
            target = signal[INPUT_SIGNAL_LEN:total]
        except Exception:
            target = np.zeros(OUTPUT_WAVEFORM_LEN)

        return (
            img_t,
            torch.tensor(target, dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def set_backbone_trainable(model, trainable):
    for p in model.encoder.vit.parameters():
        p.requires_grad = trainable
    for p in model.encoder.eff.parameters():
        p.requires_grad = trainable
    status = "UNFROZEN" if trainable else "FROZEN"
    n = sum(1 for p in model.parameters() if p.requires_grad)
    logger.info(f"[OK] Backbones {status} | Trainable params: {n:,}")


def run_epoch(model, loader, optimizer=None, training=True):
    model.train() if training else model.eval()
    total_loss, count = 0.0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch_idx, (imgs, tgts, lbls) in enumerate(
            tqdm(loader, desc="Train" if training else "Val ",
                 disable=not USE_PROGRESS_BARS, leave=False)
        ):
            imgs = imgs.to(DEVICE)
            tgts = tgts.to(DEVICE)
            lbls = lbls.to(DEVICE)

            w_pred, logits = model(imgs)
            loss = (LOSS_ALPHA * F.mse_loss(w_pred, tgts) +
                    LOSS_BETA  * F.cross_entropy(logits, lbls))

            if training:
                (loss / ACCUMULATION_STEPS).backward()
                if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                    nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                if (batch_idx + 1) % MEMORY_CLEANUP_FREQ == 0:
                    gc.collect()

            total_loss += loss.item()
            count += 1

    return total_loss / max(count, 1)


def save_loss_curves(train_losses, val_losses, save_path):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Train Loss", marker="o")
    plt.plot(val_losses,   label="Val Loss",   marker="s")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close()
    logger.info(f"[OK] Loss curve saved to {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN TRAINING
# ─────────────────────────────────────────────────────────────────────────────

def train():
    logger.info("="*80)
    logger.info("ECG HYBRID AI - TWO-PHASE TRAINING")
    logger.info("="*80)
    print_config()
    verify_paths()

    # Load metadata
    logger.info("Loading dataset metadata...")
    meta = pd.read_csv(DB_CSV, index_col="ecg_id")
    meta.scp_codes = meta.scp_codes.apply(ast.literal_eval)
    scp_df = pd.read_csv(SCP_CSV, index_col=0)

    all_records = meta[meta.filename_hr.notnull()].index.tolist()
    if MAX_RECORDS:
        all_records = all_records[:MAX_RECORDS]
    logger.info(f"[OK] Total records: {len(all_records)}")

    # Train / Val / Test split
    import random
    random.seed(42)
    random.shuffle(all_records)
    n       = len(all_records)
    n_test  = int(n * TEST_SPLIT)
    n_val   = int(n * VALIDATION_SPLIT)
    n_train = n - n_val - n_test

    train_recs = all_records[:n_train]
    val_recs   = all_records[n_train:n_train + n_val]
    test_recs  = all_records[n_train + n_val:]

    logger.info(f"Split -> Train: {len(train_recs)} | Val: {len(val_recs)} | Test: {len(test_recs)}")

    # Datasets
    logger.info("Building datasets (caching labels)...")
    train_ds = ECGDataset(train_recs, meta, scp_df)
    val_ds   = ECGDataset(val_recs,   meta, scp_df)
    test_ds  = ECGDataset(test_recs,  meta, scp_df)

    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,     shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_ds,   batch_size=VAL_BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_ds,  batch_size=VAL_BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # Model
    model = build_model(eval_mode=False)

    # Phase 1 optimizer (frozen backbones)
    set_backbone_trainable(model, False)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    best_val_loss     = float("inf")
    patience_counter  = 0
    train_losses      = []
    val_losses        = []

    logger.info("="*80)
    logger.info("STARTING TRAINING")
    logger.info("="*80)

    for epoch in range(NUM_EPOCHS):
        epoch_num = epoch + 1

        # Phase transition
        if epoch == WARMUP_EPOCHS:
            logger.info("-"*60)
            logger.info("PHASE 2: Unfreezing backbones for fine-tuning")
            logger.info("-"*60)
            set_backbone_trainable(model, True)
            optimizer = optim.AdamW(
                model.parameters(), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY
            )

        phase = "Frozen" if epoch < WARMUP_EPOCHS else "Finetune"
        logger.info(f"Epoch {epoch_num}/{NUM_EPOCHS} [{phase}]")

        train_loss = run_epoch(model, train_loader, optimizer, training=True)
        val_loss   = run_epoch(model, val_loader,   training=False)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        logger.info(
            f"  Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Gap: {abs(train_loss - val_loss):.4f}"
        )

        # Overfit/underfit hint
        gap = val_loss - train_loss
        if gap > 0.1:
            logger.info(f"  [NOTE] Gap={gap:.4f} -- possible overfitting")
        elif train_loss > 0.5 and val_loss > 0.5:
            logger.info(f"  [NOTE] Both losses high -- possible underfitting")
        else:
            logger.info(f"  [NOTE] Training looks healthy")

        # Checkpoint
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            model_path = MODELS_DIR / "ecg_best.pth"
            torch.save(model.state_dict(), model_path)
            logger.info(f"  [OK] Best model saved (val={best_val_loss:.4f})")
        else:
            patience_counter += 1
            logger.info(f"  No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                logger.info(f"  Early stopping at epoch {epoch_num}")
                break

        gc.collect()

    # Save loss curves
    curve_path = MODELS_DIR / "loss_curves.png"
    save_loss_curves(train_losses, val_losses, curve_path)

    # ── Final test evaluation ──────────────────────────────────────────────
    logger.info("="*80)
    logger.info("FINAL TEST EVALUATION")
    logger.info("="*80)

    # Load best model
    best_path = MODELS_DIR / "ecg_best.pth"
    if best_path.exists():
        state = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(state)
        logger.info("[OK] Best model loaded for test evaluation")

    test_loss = run_epoch(model, test_loader, training=False)
    logger.info(f"Test Loss: {test_loss:.4f}")

    # Overfitting analysis
    final_train = train_losses[-1]
    final_val   = val_losses[-1]
    logger.info("="*80)
    logger.info("OVERFITTING / UNDERFITTING ANALYSIS")
    logger.info("="*80)
    logger.info(f"Final Train Loss : {final_train:.4f}")
    logger.info(f"Final Val Loss   : {final_val:.4f}")
    logger.info(f"Test Loss        : {test_loss:.4f}")
    logger.info(f"Train-Val Gap    : {abs(final_train - final_val):.4f}")

    if final_val > final_train + 0.1:
        logger.info("[RESULT] OVERFITTING detected -- val loss much higher than train loss")
        logger.info("  Suggestions: add dropout, reduce model size, add data augmentation")
    elif final_train > 0.5 and final_val > 0.5:
        logger.info("[RESULT] UNDERFITTING detected -- both losses are high")
        logger.info("  Suggestions: train longer, increase model capacity, lower LR")
    else:
        logger.info("[RESULT] Model is well-fitted -- train and val losses are close")

    # Save results
    results = {
        "train_losses"  : train_losses,
        "val_losses"    : val_losses,
        "test_loss"     : test_loss,
        "best_val_loss" : best_val_loss,
        "epochs_run"    : len(train_losses),
    }
    results_path = BASE_DIR / "training_results.json"
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    logger.info(f"[OK] Results saved to {results_path}")
    logger.info("[SUCCESS] TRAINING FINISHED SUCCESSFULLY!")

    return True


if __name__ == "__main__":
    try:
        train()
    except KeyboardInterrupt:
        logger.warning("Training interrupted by user")
    except Exception as e:
        logger.error(f"Fatal error: {e}", exc_info=True)
'''

with open('/kaggle/working/ecg_project/train_OPTIMIZED.py', 'w') as f:
    f.write(train_code.strip())
print("[OK] train_OPTIMIZED.py written")

In [ ]:
import os
os.chdir('/kaggle/working/ecg_project')
!python train_OPTIMIZED.py

In [ ]:
import shutil
from pathlib import Path

WORK   = Path('/kaggle/working/ecg_project')
OUTPUT = Path('/kaggle/working')

for f in list(WORK.glob('models/*.pth')) + \
         list(WORK.glob('models/*.png')) + \
         [WORK / 'training_results.json']:
    if f.exists():
        shutil.copy(f, OUTPUT / f.name)
        print(f"[OK] Saved {f.name}")

print("\n[SUCCESS] All files saved - download from Output tab")

In [8]:
import shutil, os
from pathlib import Path

WORK   = Path('/kaggle/working/ecg_project')
OUTPUT = Path('/kaggle/working')

files_to_copy = [
    'train_OPTIMIZED.py',
    'config_OPTIMIZED.py', 
    'model_OPTIMIZED.py',
    'pre_render_OPTIMIZED.py',
]

for f in files_to_copy:
    src = WORK / f
    if src.exists():
        shutil.copy(src, OUTPUT / f)
        print(f'[OK] {f}')

print('Done - check Output tab')

[OK] train_OPTIMIZED.py
[OK] config_OPTIMIZED.py
[OK] model_OPTIMIZED.py
[OK] pre_render_OPTIMIZED.py
Done - check Output tab
